# 01 — EDA and Data Cleaning

This notebook loads the Walmart weekly sales data, inspects it, checks for data quality issues, and saves a cleaned version.

**Dataset:** 45 Walmart stores, weekly sales from Feb 2010 to Oct 2012 (6,435 rows expected).

## 1. Setup — Import libraries

In [ ]:
# Import the libraries we need
import pandas as pd   # For working with tables (DataFrames)
import numpy as np    # For math operations

# So we can see all columns when printing
pd.set_option('display.max_columns', None)

## 2. Load the data (Task 1.2)

We read the CSV file and tell pandas the date format is day/month/year (`dayfirst=True`), because `Walmart.csv` uses dates like `05-02-2010` meaning February 5, 2010.

In [ ]:
# Load the raw data from CSV
# dayfirst=True tells pandas the date is in dd-mm-yyyy format
df = pd.read_csv('../data/Walmart.csv', parse_dates=['Date'], dayfirst=True)

# Quick check: print the shape (rows, columns)
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")

The dataset has **6,435 rows** and **8 columns**, exactly as expected (45 stores × 143 weeks = 6,435).

### 2.1 Data types

Let's see what type each column is. `Date` should be `datetime64`, numbers should be `float64` or `int64`.

In [ ]:
# Show the data types of each column
print(df.dtypes)

All types look correct: `Store` and `Holiday_Flag` are integers, `Date` is datetime, and the rest are floats.

### 2.2 First few rows

Let's look at the first 5 rows to see what the data looks like.

In [ ]:
# Show the first 5 rows
df.head()

The data is organized by store, then by date within each store. Weekly sales are in the hundred-thousands to millions range.

### 2.3 Basic statistics

`describe()` gives us the count, mean, std, min, max, and quartiles for every numeric column. This is a quick way to spot anything weird.

In [ ]:
# Show basic statistics for all numeric columns
df.describe()

Key observations from `describe()`:
- **Weekly_Sales** ranges from ~$210K to ~$3.8M — big variation across stores.
- **Temperature** ranges from -2°F to 100°F — reasonable for US locations.
- **Holiday_Flag** mean is 0.07, meaning only ~7% of weeks are holiday weeks.
- **CPI** ranges from 126 to 227, and **Unemployment** from 3.9% to 14.3% — both show variation across stores/regions.

### 2.4 Date range and unique stores

We check the earliest and latest dates and how many unique stores there are.

In [ ]:
# Date range
print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"Total weeks covered: {df['Date'].nunique()}")
print(f"Number of unique stores: {df['Store'].nunique()}")
print(f"Store numbers: {sorted(df['Store'].unique())}")

The data covers **Feb 5, 2010 to Oct 26, 2012** — about 2 years and 9 months. There are **143 unique weeks** and **45 stores** (numbered 1 to 45).

---
## 3. Data Cleaning Checks (Task 1.3)

Now we check for common data quality issues: missing values, duplicates, invalid values, and outliers.

### 3.1 Missing values

We check if any column has missing (NaN) values. Missing data can cause models to crash or give wrong results.

In [ ]:
# Count missing values in each column
missing = df.isnull().sum()
print("Missing values per column:")
print(missing)
print(f"\nTotal missing values: {missing.sum()}")

**Finding:** No missing values in any column. The dataset is complete — no imputation needed.

### 3.2 Duplicate rows

We check if any row appears more than once. Each store should have exactly one entry per week.

In [ ]:
# Check for duplicate rows
n_duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {n_duplicates}")

# Also check for duplicate Store+Date pairs (each store-week should be unique)
n_store_date_dupes = df.duplicated(subset=['Store', 'Date']).sum()
print(f"Duplicate Store+Date combinations: {n_store_date_dupes}")

**Finding:** No duplicate rows and no duplicate Store+Date combinations. Each store-week is unique.

### 3.3 Negative or zero sales

Sales should be positive. Negative sales could mean returns or errors. Zero sales might mean a store was closed.

In [ ]:
# Check for negative sales
n_negative = (df['Weekly_Sales'] < 0).sum()
print(f"Rows with negative Weekly_Sales: {n_negative}")

# Check for zero sales
n_zero = (df['Weekly_Sales'] == 0).sum()
print(f"Rows with zero Weekly_Sales: {n_zero}")

**Finding:** No negative or zero sales. All Weekly_Sales values are positive, which is expected for store-level weekly totals.

### 3.4 Rows per store

Each of the 45 stores should have the same number of weeks (143). If a store has fewer rows, some weeks are missing.

In [ ]:
# Count rows per store
rows_per_store = df.groupby('Store').size()

print(f"Expected rows per store: 143")
print(f"Min rows per store: {rows_per_store.min()} (Store {rows_per_store.idxmin()})")
print(f"Max rows per store: {rows_per_store.max()} (Store {rows_per_store.idxmax()})")

# Check if all stores have exactly 143 rows
all_143 = (rows_per_store == 143).all()
print(f"All stores have 143 rows: {all_143}")

**Finding:** All 45 stores have exactly 143 rows each. The data is complete and balanced — no missing weeks for any store.

### 3.5 Outlier check using IQR method

The IQR (Interquartile Range) method flags values that are unusually high or low:
- Q1 = 25th percentile, Q3 = 75th percentile
- IQR = Q3 − Q1
- Outliers are below Q1 − 1.5×IQR or above Q3 + 1.5×IQR

**Important:** We only *report* outliers here. We do NOT remove them because high sales during holidays are real demand.

In [ ]:
# IQR outlier detection for Weekly_Sales
Q1 = df['Weekly_Sales'].quantile(0.25)
Q3 = df['Weekly_Sales'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['Weekly_Sales'] < lower_bound) | (df['Weekly_Sales'] > upper_bound)]

print(f"Q1: ${Q1:,.2f}")
print(f"Q3: ${Q3:,.2f}")
print(f"IQR: ${IQR:,.2f}")
print(f"Lower bound: ${lower_bound:,.2f}")
print(f"Upper bound: ${upper_bound:,.2f}")
print(f"\nNumber of outliers: {len(outliers)} ({len(outliers)/len(df)*100:.1f}% of data)")
print(f"Outliers below lower bound: {(df['Weekly_Sales'] < lower_bound).sum()}")
print(f"Outliers above upper bound: {(df['Weekly_Sales'] > upper_bound).sum()}")

In [ ]:
# How many of the upper outliers are holiday weeks?
upper_outliers = df[df['Weekly_Sales'] > upper_bound]
holiday_outliers = upper_outliers[upper_outliers['Holiday_Flag'] == 1]
non_holiday_outliers = upper_outliers[upper_outliers['Holiday_Flag'] == 0]

print(f"Upper outliers that are holiday weeks: {len(holiday_outliers)}")
print(f"Upper outliers that are non-holiday weeks: {len(non_holiday_outliers)}")
print(f"\nDecision: Keep all outliers. Holiday spikes are real demand,")
print(f"and large non-holiday sales likely come from bigger stores.")

**Finding:** There are 34 outliers (0.5% of data), all above the upper bound ($2,720,371). None below the lower bound. Of the 34 upper outliers, 9 are holiday weeks and 25 are non-holiday weeks (likely from the largest stores). **Decision: keep all outliers** — they represent real demand from big stores and holiday periods, not data errors.

---
## 4. Save cleaned data (Task 1.4)

Since no rows needed to be removed (no missing values, no duplicates, no invalid values), the "cleaned" data is the same as the original but with the `Date` column properly parsed as datetime. We save it to `data/cleaned.csv` as a checkpoint.

In [ ]:
# Save the cleaned dataframe to CSV
# index=False means we don't save the row numbers
df.to_csv('../data/cleaned.csv', index=False)
print(f"Saved cleaned data to data/cleaned.csv ({df.shape[0]} rows, {df.shape[1]} columns)")